In [2]:
!pip install langchain_community langchain_openai faiss-cpu pypdf
!pip install python-dotenv
import os 
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 환경 변수 가져오기
API_KEY = os.getenv("API_KEY")

In [3]:
import os
import openai
from langchain.chains import AnalyzeDocumentChain
from langchain.chains.question_answering import load_qa_chain
from langchain.chat_models import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQAWithSourcesChain, LLMChain, StuffDocumentsChain
# from langchain.retrievers import EmbeddingRetriever
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.docstore.document import Document
import numpy as np
####
from dotenv import load_dotenv
import json
import faiss
from openai import OpenAI


#### api 키 설정
api_key = API_KEY
os.environ['OPENAI_API_KEY'] = api_key
####

client = OpenAI(
    api_key=api_key,
)

def generate_text(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model="o1",
        messages=[
            {"role":"system", "content":system_prompt},
            {"role": "user", "content": user_prompt}
            ]
    )
    return response.choices[0].message.content.strip()

## 논점 프롬프트 및 생성

In [7]:
k_system_text = '''
당신은 대한민국 수학능력시험 국어영역 독서 과목의 지문을 출제하는 한국교육과정평가원 출제위원이다. 
고등학교 3학년 수준의 수험생을 평가할 수 있는 지문을 아래의 핵심 논점 및 난이도 요구사항, 금지사항을 반영하여 작성하십시오.

**핵심 논점 및 난이도 요구사항**
- 주요 개념 간의 관계를 논리적으로 설명하고, 지문에 나타난 논지의 타당성을 검토할 것
- 동일한 화제에 대한 상반되거나 다양한 관점을 비교·분석하며, 각 관점의 타당성을 비판적으로 평가할 수 있도록 구성할 것
- 각 분야의 학문적 배경을 반영하되, 개념적 깊이를 확보하고, 전문 용어는 문맥 속에서 명확히 설명할 것
- 단순한 정보 전달이 아니라 수험생이 논리적 추론을 수행할 수 있도록 유도할 것

**금지 사항** 
- 모호하거나 중의적인 표현 사용 금지
- 문학 작품 생성 금지
- 허구적 사건이나 인물명 사용 금지
- 비문 생성 금지 - (가), (나), (다) 등의 기호 사용 금지
- 결론에서 전체 내용을 요약하거나 교훈 제시 금지
- 자극적이거나 선정적인 문체 사용 금지
- 특정 집단을 비하하거나 옹호하는 내용 또는 잘못된 고정관념을 유발하는 내용 작성 금지

위 조건을 철저히 준수하여 수학능력시험 국어영역 독서 과목 지문을 출제하십시오.'''

In [12]:
subject_query = "기술"
topic_query = "인공지능과 기계학습"
passage_result = ""

In [8]:
k_user_prompt = f"""
다음은 한국교육과정평가원 스타일로 생성된 수능 독서 지문입니다.

[생성된 지문]
{passage_result}

이 지문에서 학생이 반드시 이해해야 할 핵심 논점 2~3개를 요약하세요.
각 논점은 1~2문장으로 정리하고, 출제 의도를 반영해야 합니다.

"""
key_points = generate_text(k_system_text, k_user_prompt)
key_points

'1. 첫 번째 논점: 지문에서 제시된 두 관점이 서로 다른 근거를 통해 동일한 대상에 대해 판이한 해석을 제시한다는 점에 주목해야 한다. 이를 통해 학생들은 다양한 시각을 비교·분석하고, 각 이론이 가진 타당성과 한계를 파악해야 한다.\n\n2. 두 번째 논점: 특정 개념이 다른 개념과 어떠한 상호 작용을 통해 복합적인 문제 해결 방안을 모색하는지 이해해야 한다. 이 과정을 통해 학생들은 지문이 제시하는 개념적 연관성과 그 논리적 구조를 체계적으로 추론하는 능력을 기를 수 있다.'

## 문항 프롬프트 및 생성

In [9]:
q_system_text = '''

        ## 문항 유형별 정리
        ### 1. 정답형
        - 하나의 정답과 네 개의 오답으로 구성
        - 명확한 정오 판단이 가능해야 함

        #### a. 사실적 읽기
        - 윗글에 대한 이해로 적절한 것은?
        - 윗글의 내용 전개 방식으로 적절한 것은?
        - 윗글을 읽고 보인 반응으로 적절한 것은?
        - 글에서 알 수 있는 'OOO'의 생각으로 적절한 것은?
        - 밑줄에 해당하는 내용으로 적절한 것은?

        ### 2. 부정형
        - 다섯 개의 답지 중 옳지 않은 답지를 선택하는 형식
        - 부정어(않은, 아닌, 다른, 틀린 등)를 강조하여 실수를 방지해야 함

        #### a. 사실적 읽기
        - 윗글에 대한 이해로 적절하지 않은 것은?
        - 윗글의 내용 전개 방식으로 적절하지 않은 것은?
        - 윗글을 읽고 보인 반응으로 적절하지 않은 것은?
        - 글에서 알 수 있는 'OOO'의 생각으로 적절하지 않은 것은?
        - 밑줄에 해당하는 내용으로 적절하지 않은 것은?

        ## 바람직하지 않은 문항
        - 정답에 대한 이의 제기 가능성이 있는 문항
        - 부정확한 정보가 포함된 문항
        - 특정 세계관을 강요하는 문항
        - 지나치게 복잡하거나 시간이 많이 소요되는 문항
        - 언어 사용이 부자연스러운 문항
        - 선택지만 보고도 정답을 찾을 수 있는 문항

        ## 좋은 문항의 기준
        1. 측정하고자 하는 내용과 일치해야 함
        2. 복합적인 사고 능력을 평가할 수 있어야 함
        3. 요약적이며 핵심 내용을 포함해야 함
        4. 참신한 형식과 내용을 고려해야 함
        5. 구조화되어 있고 체계성이 있어야 함
        6. 난이도가 적절해야 함
        7. 학습 동기를 유발할 수 있어야 함

        ## 선택지 작성 원리
        1. 문법적, 논리적으로 발문과 일치해야 함
        2. 정답을 모르는 수험생에게도 매력적인 오답을 제공해야 함
        3. 정답 위치를 무작위로 배치하여 패턴을 피해야 함
        4. 서술 순서를 논리적으로 배열해야 함
        5. 다른 문항과 중복되지 않도록 작성해야 함
        6. 정답과 오답이 구별되면서도 유사한 수준의 매력도를 유지해야 함
        7. 특정 어휘를 대체하는 방식으로 오답을 구성하지 않도록 주의해야 함
    '''
        

In [10]:
# 1. 디렉토리 내의 모든 문제 관련 .txt 파일을 불러오기
loader_question = DirectoryLoader("./RAG자료/문제 관련", glob="*.txt", loader_cls=TextLoader)
documents_question = loader_question.load()

# 3. OpenAI 임베딩 모델 초기화
embeddings_model = OpenAIEmbeddings()

# 4.1 문제 전체 문서 FAISS 인덱스 생성
texts_question = [doc.page_content for doc in documents_question]
metadata_question = [doc.metadata for doc in documents_question]
vector_db_question_all = FAISS.from_texts(texts_question, embeddings_model, metadatas=metadata_question)

# 5. FAISS 인덱스 저장 (문항 전체)
faiss_question_all_path = "./faiss_index/faiss_index_question_all"
faiss_question_required_path = "./faiss_index/faiss_index_question_required"

vector_db_question_all.save_local(faiss_question_all_path)
print(f"문제 전체 문서 FAISS 벡터 DB 저장 완료: {faiss_question_all_path}")

문제 전체 문서 FAISS 벡터 DB 저장 완료: ./faiss_index/faiss_index_question_all


In [15]:
# 전체 문서 FAISS에서 유사한 문서 3개 검색
question_guildlines = vector_db_question_all.similarity_search(subject_query, k=1)

# 검색 결과 출력
for i, res in enumerate(question_guildlines):
    print(f"\n[{i+1}] 검색된 문서:")
    print("-" * 50)
    print(res.page_content)
    print("-" * 50)
    print("📄 Metadata:", res.metadata)



[1] 검색된 문서:
--------------------------------------------------
- 기술의 핵심 원리나 방법을 <보기>의 그림으로 제시하여, 장치나 시스템의 작동 원리에 대해 추론을 할 수 있는지를 평가히는 문항이 자주 출제되고 있다.
--------------------------------------------------
📄 Metadata: {'source': 'RAG자료\\문제 관련\\26수특_기술분야의 출제경향_문항.txt'}


In [16]:
question_type =""

#부정형/정답형 처리
if "않은" in question_type or "않는" in question_type:
    narrative_style = "부정형"
else:
    narrative_style = "정답형"

In [17]:
q_user_prompt = f"""
    다음은 문제와 선택지를 작성할 때 반드시 고려해야 할 기준이다. 문제와 선택지를 작성하는 지침에 따라 내용을 참고하여 정확히 반영하시오:
    아래 지문을 바탕으로 학생의 이해력, 논리적 추론 능력, 비판적 사고력을 평가할 수 있는 5지선다형 객관식 문항을 1개 작성하시오.

    [지문]
    {passage_result}

    [핵심 논점(출제 의도)]
    다음은 지문에서 강조되었던 핵심 논점입니다. 문항 출제 시 반드시 아래 논점을 바탕으로 학생의 이해 및 사고력을 평가할 수 있도록 하세요.
    {key_points}

    [출제 방향 설정]
    - 반드시 위 '핵심 논점'에 근거하여 문제를 출제하십시오.
    - 다음은 "{subject_query}" 관련 수학능력시험 국어영역 독서과목 출제 경향 가이드라인입니다.
      이 기준을 충실히 반영하여 문제를 작성하십시오.
		{question_guildlines}
  
    - 출제 오류가 발생하지 않도록, 선택지는 모두 지문 내용이나 논리와 명백히 연결되어야 합니다.
    
        [금지 사항]
        - 지문의 내용과 관련 없는 문제 생성 금지.
        - 비논리적이거나 두 개 이상의 답이 나올 수 있는 문제 생성 금지.
        - 문제와 정답이 명확하지 않은 경우 생성 금지.
        - 지문의 한 정보를 읽고 풀 수 있는 선택지가 2개 이상 있으면 안 됨

        [작성 조건]
        - 하나의 선택지에서는 하나의 정보만 물어볼 것.
        - 지문에 쓰인 명사는 풀어쓰거나 비슷한 표현으로 바꾸지 않고 그대로 쓸 것.
		    - 선택지의 문장은 길이에 따라 짧은 문장에서 긴 문장 순으로 배열하십시오.
		    - 선택지의 길이 정렬이 지켜지지 않으면 출제 오류로 간주합니다.
		    - 예시:
		      1. 밑줄 긋기는 독자의 기억을 돕는다.
		      2. 무분별한 밑줄 긋기는 독서 흐름을 방해할 수 있다.
		      3. 특정 정보를 강조하여 시각적 주의를 기울이도록 한다.
		      4. 너무 많은 정보를 표시하면 중요한 내용을 파악하기 어려워질 수 있다.
		      5. 효과적인 밑줄 긋기 방법은 핵심 정보만 표시하고 과도한 사용을 자제하는 것이다.

    [문제 생성 출력 형식]
    [문제 유형]
    문항 유형 사실적 읽기/추론적 읽기/비판적 읽기 중 1개
    서술 방식 {narrative_style}
    
    [논점]
    {key_points}
    
    [질문]
    {question_type}
    
    [선택지]
    1. (내용)
    2. (내용)
    3. (내용)
    4. (내용)
    5. (내용)
    
    정답: X번
    
    문제 해설: (정답의 근거와 오답이 틀린 이유를 포함한 상세 해설 최소 100자 최대 200자)
    
    """


In [ ]:
question_data = generate_text(q_system_text,q_user_prompt)
question_data